In [0]:
import os
from pyspark.sql import functions as F
from datetime import datetime as dt

In [0]:
notebook_dir = os.getcwd()
file_path = os.path.join(notebook_dir, "..", "data", "sales_source_1500.csv")

log_dir = os.path.join(notebook_dir, ".." ,"logs")
os.makedirs(log_dir, exist_ok=True)
log_file = os.path.join(log_dir, "data_quality_report.txt")

In [0]:
logMsg = []
def time():
    return dt.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:23]
logMsg.append(f"--------------------- Data Quality check run: {time()} ---------------------")

In [0]:
df = spark.read.csv(f"file:{file_path}", header=True, inferSchema=True)
# df.show()
dataCount = df.count()

In [0]:
if dataCount == 0:
    logMsg.append(f"[{time()}] CRITICAL: Data file is empty")
    with open(log_file,"a") as f:
        f.write("\n".join(logMsg) + "\n")

    raise Exception("Data file is empty")
else:
    logMsg.append(f"[{time()}]: Data file has {dataCount} rows.")
    # print(f"Data file has {dataCount} rows")

In [0]:
nullCounts = False
nullVal = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).collect()[0]
# display(nullVal)

for c in df.columns:
    if nullVal[c] > 0:
        logMsg.append(f"[{time()}]: Column '{c}' has {nullVal[c]} null values")
        # print(f"Column '{c}' has {nullVal[c]} null values")
        nullCounts = True

if not nullCounts:
    logMsg.append(f"[{time()}]: No null values found")
    # print("No null values found")

In [0]:
invalidData = df.filter( 
                        (F.col("quantity") <= 0) | 
                        (F.col("unit_price") <= 0) | 
                        (F.col("gross_amount") <= 0) | 
                        (F.col("net_amount") <= 0) |
                        (F.col("product_id") == "UNKNOWN")
                        )

if invalidData.count() > 0:
    logMsg.append(f"[{time()}]: Invalid data found: {invalidData.count()} rows")
    # print(f"Invalid data found: {invalidData.count()} rows")
else:
    logMsg.append(f"[{time()}]: No invalid data found")
    # print("No invalid data found")

In [0]:
logMsg.append(f"--------------------- Data Quality check end: {time()} ---------------------")

with open(log_file,"a") as f:
    f.write("\n".join(logMsg) + "\n")
# print("\n".join(logMsg))